In [1]:
# ==============================================================================
# ENTERPRISE P&L PIPELINE: GOLD LAYER (ANALYTICS & DIMENSIONAL MODELING)
# ==============================================================================
import uuid
import traceback
from datetime import datetime
from zoneinfo import ZoneInfo

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, coalesce, trim, lower, lit, from_utc_timestamp, substring, 
    split, row_number, when, concat, concat_ws, to_date, lead
)
from pyspark.sql.window import Window

# ------------------------------------------------------------------------------
# 1. PIPELINE CONFIGURATION & SPARK INITIALIZATION
# ------------------------------------------------------------------------------
class PipelineConfig:
    PIPELINE_NAME      = "NB_Corp_Finance_DataHub_ELT_Gold_Profitability" 
    ENVIRONMENT        = "Production"
    TZ                 = "America/New_York"
    RUN_ID             = str(uuid.uuid4())
    TARGET_CATALOG     = "LH_Corp_DataHub"
    
    # POWER BI SEMANTIC MODEL
    SEMANTIC_MODEL     = "SM_Corp_Project_Real_Profitability" 
    
    # Schema Definitions
    SCHEMA_SILVER      = "silver"
    SCHEMA_DATAVERSE   = "dataverse"         
    SCHEMA_DV_HISTORY  = "dataverse_history" 
    SCHEMA_HR          = "hr"                
    SCHEMA_GOLD        = "gold"

# Initialize Spark Session with strict schema and calendar configurations
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# [FIX] Handle legacy dates (pre-1582) found in UKG raw data to avoid write failures
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# ------------------------------------------------------------------------------
# 2. TELEMETRY & UTILITIES
# ------------------------------------------------------------------------------
def log_info(msg):    print(f"{datetime.now(ZoneInfo(PipelineConfig.TZ)).strftime('%Y-%m-%d %H:%M:%S')} - [INFO] - {msg}")
def log_error(msg):   print(f"{datetime.now(ZoneInfo(PipelineConfig.TZ)).strftime('%Y-%m-%d %H:%M:%S')} - [ERROR] - {msg}")
def log_success(msg): print(f"{datetime.now(ZoneInfo(PipelineConfig.TZ)).strftime('%Y-%m-%d %H:%M:%S')} - [SUCCESS] - {msg}")

def add_metadata(df: DataFrame) -> DataFrame:
    return (df.withColumn("_gold_modified_at", from_utc_timestamp(F.current_timestamp(), PipelineConfig.TZ))
              .withColumn("_gold_pipeline_run_id", lit(PipelineConfig.RUN_ID)))

# ------------------------------------------------------------------------------
# 3. CORE TRANSFORMATION ENGINE (GOLD LAYER)
# ------------------------------------------------------------------------------
def build_gold_profitability_fact():
    """
    [Gold Transformation] Advanced Analytics Engine (Fact Table)
    - Implements Dynamic Dimension Routing (Umbrella Projects logic).
    - Computes Intercompany and Cross-Charge analytical flows based on Org hierarchy.
    - Includes 'org_key' to bridge cross-charge transactions in Power BI with dim_organization.
    """
    log_info("Reading Silver Fact Table (Source of Truth)...")
    df_silver = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_SILVER}.fact_time_and_cost_enriched")
    
    # --- DIMENSION 1: MASTER PROJECTS ---
    df_proj_mst = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DV_HISTORY}.msdyn_project") \
        .select(
            col("msdyn_projectid").alias("dv_project_id"), 
            col("msdyn_subject").alias("mst_name"),
            col("msdyn_contractorganizationalunitidname").alias("project_company"),
            col("msdyn_comments"), 
            col("aux_customername").alias("mst_customer"),
            col("aux_industryname").alias("mst_industry"),
            col("aux_servicelinename").alias("mst_service_line"),
            col("aux_organizationname")
        ).withColumn("mst_L1", split(col("aux_organizationname"), "-").getItem(0)) \
         .withColumn("mst_L2", split(col("aux_organizationname"), "-").getItem(1)) \
         .withColumn("mst_L3", split(col("aux_organizationname"), "-").getItem(2)) \
         .drop("aux_organizationname") \
         .dropDuplicates(["dv_project_id"])

    # --- DIMENSION 2: FINANCIAL FALLBACK ---
    df_financials = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.fact_financials_unified") \
        .withColumn("fin_proj_key", substring(col("project_name"), 1, 10))
    
    window_spec = Window.partitionBy("fin_proj_key").orderBy(col("transaction_date").desc())
    df_proj_fin = df_financials.withColumn("rn", row_number().over(window_spec)) \
        .filter(col("rn") == 1) \
        .select(
            col("fin_proj_key"),
            col("aux_customername").alias("fin_customer"),
            col("aux_industryname").alias("fin_industry"),
            col("aux_servicelinename").alias("fin_service_line"),
            col("reporting_org").alias("fin_company"),
            col("L1").alias("fin_L1"), col("L2").alias("fin_L2"), col("L3").alias("fin_L3")
        )

    # --- DIMENSION 3: TASK DIMENSIONS (UMBRELLA) ---
    df_task = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DV_HISTORY}.msdyn_projecttask") \
        .select(
            col("msdyn_projecttaskid").alias("tid"), col("msdyn_subject").alias("task_name"),
            col("aux_customername").alias("tsk_customer"),
            col("aux_industryname").alias("tsk_industry"),
            col("aux_servicelinename").alias("tsk_service_line"),
            col("aux_organizationname")
        ).withColumn("tsk_L1", split(col("aux_organizationname"), "-").getItem(0)) \
         .withColumn("tsk_L2", split(col("aux_organizationname"), "-").getItem(1)) \
         .withColumn("tsk_L3", split(col("aux_organizationname"), "-").getItem(2)) \
         .drop("aux_organizationname") \
         .dropDuplicates(["tid"])

    # --- DIMENSION 4: RESOURCES ---
    df_res = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.bookableresource_clean") \
        .select(col("bookableresourceid").alias("rid"), col("name").alias("dv_resource_name")).dropDuplicates(["rid"])

    # --- JOINING & PROJECTION ---
    df_stg1 = df_silver.alias("sil") \
        .join(df_proj_mst.alias("mst"), lower(trim(col("sil.projectid"))) == lower(trim(col("mst.dv_project_id"))), "left") \
        .withColumn("p_name", coalesce(col("mst.mst_name"), lit("Unmapped Project"))) \
        .withColumn("p_key", substring(col("p_name"), 1, 10))
        
    df_final = df_stg1.alias("stg") \
        .join(df_proj_fin.alias("fin"), col("stg.p_key") == col("fin.fin_proj_key"), "left") \
        .join(df_task.alias("tsk"), lower(trim(col("stg.projecttaskid"))) == lower(trim(col("tsk.tid"))), "left") \
        .join(df_res.alias("res"), lower(trim(col("stg.bookableresourceid"))) == lower(trim(col("res.rid"))), "left")

    df_resolved = df_final.select(
        col("stg.transaction_date"), col("stg.worked_hours"), col("stg.actual_cost_usd"),
        col("stg.synerion_emp_id"), col("stg.ukg_employee_id"),
        coalesce(col("res.dv_resource_name"), col("stg.employee_full_name")).alias("employee_full_name"),
        col("stg.employee_country"), col("stg.employee_reporting_category"),
        col("stg.L1").alias("employee_L1"), col("stg.L2").alias("employee_L2"), col("stg.L3").alias("employee_L3"),
        col("stg.p_key").alias("project_key"), col("stg.p_name").alias("project_name"),
        coalesce(col("stg.project_company"), lit("N/A")).alias("project_company"),
        
        # Waterfall dimension resolution for Umbrella Projects
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_customer"), lit("N/A"))) \
            .otherwise(coalesce(col("stg.mst_customer"), col("fin.fin_customer"), lit("N/A"))).alias("project_customer"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_industry"), lit("N/A"))) \
            .otherwise(coalesce(col("stg.mst_industry"), col("fin.fin_industry"), lit("N/A"))).alias("project_industry"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_service_line"), lit("N/A"))) \
            .otherwise(coalesce(col("stg.mst_service_line"), col("fin.fin_service_line"), lit("N/A"))).alias("project_service_line"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_L1"), lit("Corporate"))) \
            .otherwise(coalesce(col("stg.mst_L1"), col("fin.fin_L1"), lit("Corporate"))).alias("project_L1"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_L2"), lit("Corporate"))) \
            .otherwise(coalesce(col("stg.mst_L2"), col("fin.fin_L2"), lit("Corporate"))).alias("project_L2"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_L3"), lit("Corporate"))) \
            .otherwise(coalesce(col("stg.mst_L3"), col("fin.fin_L3"), lit("Corporate"))).alias("project_L3"),
        
        coalesce(col("tsk.task_name"), lit("Unmapped Task")).alias("task_name"),
        col("stg.subtask_category"),
        col("stg.pay_category_id"),
        col("stg.pay_category_description")
    )

    # --- ADVANCED ANALYTICS (FLOWS) ---
    df_analytics = df_resolved.withColumn(
        "project_country",
        when(col("project_company").contains("Costa Rica"), "Costa Rica")
        .when(col("project_company").contains("Colombia"), "Colombia")
        .otherwise("United States") 
    )
    
    emp_iso = when(col("employee_country") == "Costa Rica", "CR").when(col("employee_country") == "Colombia", "COL").when(col("employee_country") == "United States", "USA").otherwise("UNK")
    prj_iso = when(col("project_country") == "Costa Rica", "CR").when(col("project_country") == "Colombia", "COL").when(col("project_country") == "United States", "USA").otherwise("UNK")
    
    df_final_model = df_analytics.withColumn(
        "intercompany_flag", when(col("employee_country") == col("project_country"), "Local").otherwise("Intercompany")
    ).withColumn(
        "intercompany_flow", when(col("intercompany_flag") == "Local", concat(emp_iso, lit(" Local"))).otherwise(concat(emp_iso, lit(" -> "), prj_iso))
    ).withColumn(
        "cross_charge_flag", when(col("employee_L1") == col("project_L1"), "Direct").otherwise("Cross Charge")
    ).withColumn(
        "cross_charge_flow", when(col("cross_charge_flag") == "Direct", concat(col("employee_L1"), lit(" Direct"))).otherwise(concat(col("employee_L1"), lit(" -> "), col("project_L1")))
    ).withColumn(
        # -------------------------------------------------------------------------
        # >> PASO 3: LLAVE DE CONEXIÓN A LA DIMENSIÓN DE ORGANIZACIÓN <<
        # -------------------------------------------------------------------------
        "org_key", F.concat_ws("|", col("project_L1"), col("project_L2"), col("project_L3"))
    )

    return df_final_model

def build_gold_dim_employee(df_fact_profitability: DataFrame):
    """
    [Gold Transformation] Static Employee Dimension (SCD Type 1)
    - Captures Master Employee details, dynamically handling LATAM custom fields.
    - Resolves missing US employee names by leveraging the Dataverse integration 
      already present in the Profitability Fact Table.
    - Sanitizes legacy effective dates to ensure Power BI semantic model stability.
    """
    log_info("Building Employee Dimension from UKG Master & Resolving Missing Names...")
    df_ukg = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.ukg_employment_details")
    
    # 1. Extract resolved names (Dataverse-enriched) from the Profitability Fact Table
    df_resolved_names = df_fact_profitability.select(
        col("ukg_employee_id").alias("fact_ukg_id"),
        col("employee_full_name").alias("resolved_name")
    ).filter(col("fact_ukg_id").isNotNull() & col("resolved_name").isNotNull()) \
     .dropDuplicates(["fact_ukg_id"])
    
    # 2. Join the UKG master dataset with the resolved names lookup
    df_joined = df_ukg.join(df_resolved_names, df_ukg.employeeid == df_resolved_names.fact_ukg_id, "left")
    
    return df_joined.select(
        col("employeeid").alias("ukg_employee_id"),
        col("employeenumber").alias("employee_number"),
        
        # Smart Name Resolution: 
        # Prioritizes the Dataverse-enriched name. If null, falls back to LATAM custom fields in UKG.
        coalesce(
            col("resolved_name"),
            when(
                col("p_config_name").isNotNull() & (trim(col("p_config_name")) != ""), 
                concat_ws(" ", col("p_config_name"), col("p_config_lastname1"))
            )
        ).alias("employee_full_name"),
        
        col("companyname").alias("legal_entity"),
        
        # Date Sanitization: Replaces invalid legacy dates (pre-1900) to prevent Power BI refresh errors
        when(to_date(col("lasthiredate")) < "1900-01-01", to_date(lit("1900-01-01")))
            .otherwise(to_date(col("lasthiredate"))).alias("hire_date"),
            
        when(to_date(col("terminationdate")) < "1900-01-01", to_date(lit("1900-01-01")))
            .otherwise(to_date(col("terminationdate"))).alias("termination_date"),
            
        col("employeestatuscode").alias("status_code")
    ).dropDuplicates(["ukg_employee_id"])

def build_gold_dim_employee_history():
    """
    [Gold Transformation] Employee History Dimension (SCD Type 2)
    - Maintains organizational hierarchy shifts over time.
    - [FIX]: Sanitizes ancient effective dates to protect Power BI engine.
    """
    log_info("Building Employee History SCD2 from UKG...")
    df_history = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.ukg_employee_job_history")
    df_dim_lookup = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.ukg_dimensions_details")
    
    # 1. Extraer y sanear fecha original
    df_scd2 = df_history.withColumn(
        "raw_date", coalesce(col("jobEffectiveDate").cast("date"), to_date(col("jobEffectiveDate"), "yyyy-MM-dd HH:mm:ss"))
    ).withColumn(
        # [FIX] Si es del año de la pera (ej. 0001-01-01), la fijamos en 1900-01-01
        "effective_date", 
        when(col("raw_date") < "1900-01-01", to_date(lit("1900-01-01"))).otherwise(col("raw_date"))
    ).withColumn(
        "end_date", 
        coalesce(lead("effective_date").over(Window.partitionBy("employeeId").orderBy("effective_date")), to_date(lit("2099-12-31"), "yyyy-MM-dd"))
    )
    
    dim_l1 = df_dim_lookup.filter(col("level") == 1).select(col("code").alias("c1"), col("description").alias("d1"))
    
    return df_scd2.alias("hist") \
        .join(dim_l1, lower(trim(col("hist.orgLevel1Code"))) == lower(trim(col("c1"))), "left") \
        .withColumn("L1", split(col("d1"), "-").getItem(0)) \
        .select(
            col("employeeId").alias("ukg_employee_id"),
            col("L1"),
            col("effective_date"),
            col("end_date")
        ).filter(col("ukg_employee_id").isNotNull())

def build_gold_refresh_info():
    """
    [Gold Transformation] Pipeline Execution Metadata
    - Captures the exact completion timestamp for front-end reporting visuals.
    """
    log_info("Generating Last Refresh Date metadata...")
    return spark.sql(f"SELECT from_utc_timestamp(current_timestamp(), '{PipelineConfig.TZ}') as LastRefresh")

def build_gold_dim_project(df_fact_profitability: DataFrame) -> DataFrame:
    """
    [Gold Transformation] Pure Project Dimension
    - Contains ONLY project-level attributes (ID, Name, Customer).
    - Organizational hierarchies (L1/L2/L3) have been moved to fact tables and a conformed org dimension
      to support multi-department billing (cross-charges) without granularity loss.
    """
    print("Building Pure Project Dimension (No Hierarchies)...")

    # 1. Operational Projects
    df_ops = df_fact_profitability.select(
        col("project_key"),
        col("project_name").alias("ops_name"),
        col("project_customer").alias("ops_customer")
    ).dropDuplicates(["project_key"]).filter(col("project_key").isNotNull())

    # 2. Financial Projects (SSOT)
    df_fin = spark.table("LH_Corp_DataHub.dataverse.fact_financials_unified").select(
        col("project_key"),
        col("Project_Name").alias("fin_name"),            
        col("aux_customername").alias("fin_customer")
    ).dropDuplicates(["project_key"]).filter(col("project_key").isNotNull())

    # 3. Full Outer Join
    df_joined = df_fin.join(df_ops, on="project_key", how="full_outer")

    # 4. Final Clean Projection
    df_dim_final = df_joined.select(
        col("project_key"),
        coalesce(col("fin_name"), col("ops_name"), col("project_key")).alias("project_name"),
        coalesce(col("fin_customer"), col("ops_customer"), lit("Unallocated Customer")).alias("project_customer")
    )

    return df_dim_final

def build_gold_revenue_fact():
    """
    [Gold Transformation] Revenue Fact Table
    - Extracts Actuals and Backlog records from Dataverse financials.
    - Excludes 'LICS' (Licenses) from L3 hierarchy to isolate services revenue.
    - Granularity: Project-Level Revenue for Financial Margin Analysis.
    - Generates 'org_key' to bridge cross-charge transactions via the Conformed Organization Dimension.
    """
    log_info("Building Revenue Fact Table (Actuals & Backlog)...")
    df_fin = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.fact_financials_unified")
    
    # 1. Filter out Licenses and include both Actuals and Backlog scenarios
    df_rev = df_fin.filter(
        (col("scenario").isin("Actual", "Backlog")) & 
        (col("L3") != "LICS")
    ).select(
        to_date(col("transaction_date")).alias("transaction_date"),
        col("project_key"),
        
        # Financial Metrics
        col("amount").cast("double").alias("actual_revenue_usd"),
        col("hours_quantity").cast("double").alias("hours_quantity"),
        col("scenario"),
        col("type").alias("revenue_type"),
        
        # Organizational & Client Hierarchies
        col("L1").alias("financial_L1"),
        col("L2").alias("financial_L2"),
        col("L3").alias("financial_L3"),
        col("aux_customername").alias("financial_customer"),
        
        # -------------------------------------------------------------------------
        # >> CONNECTION KEY TO THE CONFORMED ORGANIZATION DIMENSION <<
        # -------------------------------------------------------------------------
        F.concat_ws("|", col("L1"), col("L2"), col("L3")).alias("org_key")
    )
    
    return df_rev

def build_gold_dim_organization() -> DataFrame:
    """
    [Gold Transformation] Conformed Organization Dimension
    - Extracts unique L1, L2, L3 combinations from both financial and operational fact tables.
    - Creates a composite 'org_key' to bridge cross-charge transactions in Power BI.
    """
    print("Building Organization Dimension (L1, L2, L3)...")
    
    # 1. Extraemos las combinaciones únicas de la tabla operativa (UKG/Synerion)
    df_ops_org = spark.table("LH_Corp_DataHub.gold.fact_profitability_metrics").select(
        col("project_L1").alias("L1"),
        col("project_L2").alias("L2"),
        col("project_L3").alias("L3")
    )
    
    # 2. Extraemos las combinaciones únicas de la tabla financiera (Revenue)
    df_fin_org = spark.table("LH_Corp_DataHub.dataverse.fact_financials_unified").select(
        col("L1"),
        col("L2"),
        col("L3")
    )
    
    # 3. Unimos ambas listas y eliminamos duplicados
    df_org_union = df_ops_org.union(df_fin_org).dropDuplicates().fillna("Unallocated")
    
    # 4. Creamos la llave maestra (Concatenando L1|L2|L3) para conectar en Power BI
    df_dim_org = df_org_union.withColumn(
        "org_key", 
        F.concat_ws("|", col("L1"), col("L2"), col("L3"))
    )
    
    return df_dim_org.select("org_key", "L1", "L2", "L3")

def build_gold_dim_hierarchy_master():
    """
    [Gold Transformation] Master Organizational Hierarchy Dimension
    - Combines unique structural hierarchies (L1, L2, L3) from both 
      Operational (Profitability) and Financial (Revenue) fact tables.
    - Filters data from '2026-01-01' onwards to ensure historical cleanliness.
    - Prevents blind spots caused by cross-charged resources or shared projects.
    """
    log_info("Building Unified Master Hierarchy Dimension (Profitability + Revenue)...")
    
    # 1. Extract distinct hierarchy from Operational Metrics (Synerion/UKG)
    df_profitability = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_profitability_metrics") \
                            .filter(col("transaction_date") >= "2026-01-01") \
                            .select(
                                col("employee_L1").alias("L1"), 
                                col("employee_L2").alias("L2"), 
                                col("employee_L3").alias("L3")
                            )
    
    # 2. Extract distinct hierarchy from Financial Metrics (Revenue)
    # [FIX] Usando transaction_date y los campos correctos de financial_L
    df_revenue = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_revenue_metrics") \
                      .filter(col("transaction_date") >= "2026-01-01") \
                      .select(
                          col("financial_L1").alias("L1"), 
                          col("financial_L2").alias("L2"), 
                          col("financial_L3").alias("L3")
                      )
    
    # 3. Union both sources to capture every single active combination
    df_combined = df_profitability.union(df_revenue)
    
    # 4. Clean up nulls, empty strings, and eliminate duplicates
    df_hierarchy_final = df_combined.filter(col("L1").isNotNull() & (trim(col("L1")) != "")) \
                                    .dropDuplicates(["L1", "L2", "L3"])
    
    return df_hierarchy_final

# ------------------------------------------------------------------------------
# 4. AUTOMATION & EXECUTION
# ------------------------------------------------------------------------------
def refresh_powerbi_model(dataset_name: str):
    try:
        import sempy.fabric as fabric
        log_info(f"Triggering refresh for: '{dataset_name}'...")
        fabric.refresh_dataset(dataset=dataset_name)
        log_success("Semantic Model refresh triggered.")
    except Exception as e:
        log_error(f"PBI Refresh failed: {str(e)[:200]}")

if __name__ == "__main__":
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}")
        
        # 1. Fact Table Sync (Cost y Hrs)
        df_fact = build_gold_profitability_fact()
        target_fact = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_profitability_metrics"
        log_info(f"Writing to: {target_fact}...")
        df_fact.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_fact)
        spark.sql(f"OPTIMIZE {target_fact}")
        
        # 2. Static Employee Dimension Sync (Enhanced with Fact Table dependency for name resolution)
        df_dim = build_gold_dim_employee(df_fact) 
        target_dim = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_employee"
        
        log_info(f"Writing to: {target_dim}...")
        
        # Write the dataframe as a Delta table, overwriting the existing schema if necessary
        df_dim.transform(add_metadata) \
              .write \
              .format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(target_dim)
        
        # Optimize the Delta table for faster read performance in Direct Lake mode
        spark.sql(f"OPTIMIZE {target_dim}")

        # 3. Employee History (SCD2) Sync
        df_hist = build_gold_dim_employee_history()
        target_hist = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_employee_history"
        log_info(f"Writing to: {target_hist}...")
        df_hist.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_hist)
        spark.sql(f"OPTIMIZE {target_hist}")

        # 4. Project Dimension Sync
        df_dim_proj = build_gold_dim_project(df_fact) 
        target_dim_proj = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_project"
        log_info(f"Writing to: {target_dim_proj}...")
        df_dim_proj.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_dim_proj)
        spark.sql(f"OPTIMIZE {target_dim_proj}")
        
        # 5. Revenue Fact Table Sync
        df_rev = build_gold_revenue_fact()
        target_rev = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_revenue_metrics"
        log_info(f"Writing to: {target_rev}...")
        df_rev.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_rev)
        spark.sql(f"OPTIMIZE {target_rev}")
        
        # 6. Refresh Metadata Sync
        df_refresh = build_gold_refresh_info()
        target_refresh = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.info_gold_profitability_refresh"
        log_info(f"Writing to: {target_refresh}...")
        df_refresh.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_refresh)
        
        # 7. Generate the master department hierarchy list for cross-functional reporting
        df_hierarchy = build_gold_dim_hierarchy_master()
        target_hierarchy_master = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_Hierarchy_Master"
        
        log_info(f"Writing to: {target_hierarchy_master}...")
        
        # Write the dataframe as a Delta table, overwriting the existing schema if necessary
        df_hierarchy.transform(add_metadata) \
                    .write \
                    .format("delta") \
                    .mode("overwrite") \
                    .option("overwriteSchema", "true") \
                    .saveAsTable(target_hierarchy_master)
        
        # Optimize the Delta table for faster read performance in Direct Lake mode
        spark.sql(f"OPTIMIZE {target_hierarchy_master}")

        # 7.5 Generate Conformed Organization Dimension (con org_key)
        df_organization = build_gold_dim_organization()
        target_org = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_organization"
        
        log_info(f"Writing to: {target_org}...")
        
        # Escribimos la tabla en formato Delta
        df_organization.transform(add_metadata) \
                       .write \
                       .format("delta") \
                       .mode("overwrite") \
                       .option("overwriteSchema", "true") \
                       .saveAsTable(target_org)
        
        # Optimizamos para Power BI
        spark.sql(f"OPTIMIZE {target_org}")

        # 8. Semantic Model Refresh
        refresh_powerbi_model(PipelineConfig.SEMANTIC_MODEL)
        log_success("Gold Pipeline execution complete.")
        
    except Exception as ex:
        log_error(f"Critical Failure: {str(ex)[:500]}")
        raise

StatementMeta(, 08087887-3068-4451-819b-e6773833edd6, 3, Finished, Available, Finished, False)

2026-11-03 11:41:09 - [INFO] - Reading Silver Fact Table (Source of Truth)...
2026-11-03 11:41:13 - [INFO] - Writing to: LH_Corp_DataHub.gold.fact_profitability_metrics...
2026-11-03 11:41:39 - [INFO] - Building Employee Dimension from UKG Master & Resolving Missing Names...
2026-11-03 11:41:39 - [INFO] - Writing to: LH_Corp_DataHub.gold.dim_employee...
2026-11-03 11:41:58 - [INFO] - Building Employee History SCD2 from UKG...
2026-11-03 11:41:59 - [INFO] - Writing to: LH_Corp_DataHub.gold.dim_employee_history...
Building Pure Project Dimension (No Hierarchies)...
2026-11-03 11:42:07 - [INFO] - Writing to: LH_Corp_DataHub.gold.dim_project...
2026-11-03 11:42:14 - [INFO] - Building Revenue Fact Table (Actuals & Backlog)...
2026-11-03 11:42:14 - [INFO] - Writing to: LH_Corp_DataHub.gold.fact_revenue_metrics...
2026-11-03 11:42:26 - [INFO] - Generating Last Refresh Date metadata...
2026-11-03 11:42:26 - [INFO] - Writing to: LH_Corp_DataHub.gold.info_gold_profitability_refresh...
2026-11-03